In [ ]:
import re
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import numpy as np
import torch
import yake
from pathlib import Path
from docx import Document
from tokenizers import Tokenizer
from transformers import AutoModel
from huggingface_hub import hf_hub_download
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer, models
import gc
from transformers import MarianMTModel, MarianTokenizer
from transformers import pipeline
from tqdm import tqdm
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import spacy
nlp_spacy = spacy.load("lt_core_news_sm")
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    get_scheduler)
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import time

In [ ]:
ROOT_2016 = Path(r"C:\Users\Arnas\Downloads\stenogramos\2016 - 2020 metų kadencija")
ROOT_2020 = Path(r"C:\Users\Arnas\Downloads\stenogramos\2020 - 2024 metų kadencija")

OUTPUT_DIR = Path(r"C:\Users\Arnas\Downloads\stenogramos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_docx_text(docx_path: Path) -> str:
    """
    Nuskaito tik .docx pastraipų tekstą.
    Lentelės nėra nuskaitomos.
    """
    doc = Document(docx_path)
    parts = []

    for p in doc.paragraphs:
        text = p.text.strip()
        if text:
            parts.append(text)

    return "\n".join(parts).strip()


def extract_posedzio_nr(filename: str):
    """
    Iš failo pavadinimo, pvz. 'Posėdis+Nr.29.docx', ištraukia 29.
    Veikia ir jei yra lietuviškos/nelietuviškos raidės ar smulkūs formatavimo skirtumai.
    """
    pattern = r"pos[eė]dis\+?nr\.?\s*(\d+)"
    m = re.search(pattern, filename, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def extract_session_order(folder_name: str):
    """
    Iš aplanko pavadinimo ištraukia sesijos numerį ir tipą.
    Pvz.:
    '1 eilinė Seimo sesija (2016-11-14 - 2017-01-17)' -> (1, 'eilinė')
    '7 neeilinė Seimo sesija' -> (7, 'neeilinė')
    """
    pattern = r"(\d+)\s+(eilinė|neeilinė)"
    m = re.search(pattern, folder_name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1)), m.group(2).lower()
    return None, None

In [ ]:
def clean_session_name(folder_name: str) -> str:
    """
    Pašalina datas skliaustuose iš aplanko pavadinimo pabaigos, jei jos yra.
    """
    return re.sub(r"\s*\(.*?\)\s*$", "", folder_name).strip()


def process_kadencija(root_dir: Path, kadencija: str) -> pd.DataFrame:
    """
    Nuskaito vienos kadencijos visų sesijų .docx stenogramas į DataFrame.
    """
    records = []

    if not root_dir.exists():
        print(f"Klaida: aplankas nerastas -> {root_dir}")
        return pd.DataFrame()

    session_dirs = [p for p in root_dir.iterdir() if p.is_dir()]

    all_docx_files = []
    for session_dir in session_dirs:
        for docx_file in session_dir.glob("*.docx"):
            all_docx_files.append((session_dir, docx_file))

    print(f"\nApdorojama kadencija: {kadencija}")
    print(f"Rasta sesijų aplankų: {len(session_dirs)}")
    print(f"Rasta .docx failų: {len(all_docx_files)}")

    for session_dir, docx_file in tqdm(all_docx_files, desc=f"Nuskaitomos stenogramos {kadencija}"):
        session_name_raw = session_dir.name
        session_name = clean_session_name(session_name_raw)
        session_order, session_type = extract_session_order(session_name)

        posedzio_nr = extract_posedzio_nr(docx_file.name)

        try:
            text = read_docx_text(docx_file)
            error_msg = None
        except Exception as e:
            text = None
            error_msg = str(e)

        records.append({
            "kadencija": kadencija,
            "sesijos_aplankas": session_name_raw,
            "sesija": session_name,
            "sesijos_nr": session_order,
            "sesijos_tipas": session_type,
            "failo_pavadinimas": docx_file.name,
            "posedzio_nr": posedzio_nr,
            "failo_kelias": str(docx_file),
            "tekstas": text,
            "teksto_ilgis": len(text) if text else 0,
            "klaida": error_msg
        })

    df = pd.DataFrame(records)

    if df.empty:
        print(f"Kadencijai {kadencija} nerasta duomenų.")
        return df

    df = df.sort_values(
        by=["posedzio_nr", "sesijos_nr", "failo_pavadinimas"],
        na_position="last"
    ).reset_index(drop=True)

    print(f"Sėkmingai apdorota failų: {len(df)}")
    return df

In [ ]:
def check_numbering(df: pd.DataFrame, kadencija: str):
    """
    Patikrina posėdžių numerių seką.
    """
    if df.empty:
        print(f"\n{kadencija}: duomenų nėra.")
        return

    valid_nums = df["posedzio_nr"].dropna().astype(int).sort_values().tolist()

    if valid_nums:
        expected = list(range(min(valid_nums), max(valid_nums) + 1))
        missing = sorted(set(expected) - set(valid_nums))
        duplicates = (df["posedzio_nr"][df["posedzio_nr"]
        .duplicated(keep=False)]
        .dropna()
        .astype(int)
        .tolist())

        print(f"\nNumeracijos patikra: {kadencija}")
        print(f"Mažiausias posėdžio nr.: {min(valid_nums)}")
        print(f"Didžiausias posėdžio nr.: {max(valid_nums)}")
        print(f"Trūkstami numeriai: {missing if missing else 'nėra'}")
        print(f"Besidubliuojantys numeriai: {sorted(set(duplicates)) if duplicates else 'nėra'}")
    else:
        print(f"\n{kadencija}: nepavyko rasti nė vieno posėdžio numerio.")


def save_outputs(df: pd.DataFrame, parquet_path: Path, csv_path: Path, label: str):
    """
    Išsaugo DataFrame į parquet ir csv.
    """
    if df.empty:
        print(f"{label}: nieko nesaugoma, nes DataFrame tuščias.")
        return

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\nIšsaugota: {label}")
    print(parquet_path)
    print(csv_path)

df_2016 = process_kadencija(ROOT_2016, "2016-2020")
df_2020 = process_kadencija(ROOT_2020, "2020-2024")

save_outputs(
    df_2016,
    OUTPUT_DIR / "seimo_stenogramos_2016_2020.parquet",
    OUTPUT_DIR / "seimo_stenogramos_2016_2020.csv",
    "2016-2020"
)

save_outputs(
    df_2020,
    OUTPUT_DIR / "seimo_stenogramos_2020_2024.parquet",
    OUTPUT_DIR / "seimo_stenogramos_2020_2024.csv",
    "2020-2024"
)

save_outputs(
    df_all,
    OUTPUT_DIR / "seimo_stenogramos_2016_2024.parquet",
    OUTPUT_DIR / "seimo_stenogramos_2016_2024.csv",
    "2016-2024"
)

In [ ]:
df = pd.read_parquet("seimo_stenogramos_2016_2024.parquet")
print(df.columns)
print(df.head())

df['zodziu_sk'] = df['tekstas'].apply(lambda x: len(str(x).split()))

df['sakiniu_sk'] = df['tekstas'].apply(
    lambda x: len(re.findall(r'[.!?]+', str(x)))
)

# Žodžių kiekio mediana stenogramose
kadenciju_lentele = df.groupby('kadencija').agg(
    stenogramu_sk=('tekstas', 'count'),
    zodziu_sk=('zodziu_sk', 'sum'),
    vid_zodziu_stenogramoje=('zodziu_sk', 'mean'),
    med_zodziu_stenogramoje=('zodziu_sk', 'median'),
    sakiniu_sk=('sakiniu_sk', 'sum')
)

kadenciju_lentele['vid_sakinio_ilgis'] = (
    kadenciju_lentele['zodziu_sk'] / kadenciju_lentele['sakiniu_sk']
)

# Funkcija sakinių ilgiams suskaičiuoti
def sakiniu_ilgiai(text):
    sakiniai = re.split(r'[.!?]+', str(text))
    sakiniai = [s.strip() for s in sakiniai if s.strip()]
    return [len(s.split()) for s in sakiniai]

# Sakinio ilgio mediana pagal kadenciją
sakiniu_medianos = (
    df.groupby('kadencija')['tekstas']
    .apply(lambda tekstai: np.median([
        ilgis
        for tekstas in tekstai
        for ilgis in sakiniu_ilgiai(tekstas)
    ]) if any(sakiniu_ilgiai(t) for t in tekstai) else np.nan)
)

kadenciju_lentele['med_sakinio_ilgis'] = sakiniu_medianos

visi_sakiniu_ilgiai = [
    ilgis
    for tekstas in df['tekstas']
    for ilgis in sakiniu_ilgiai(tekstas)
]

bendra_eilute = pd.DataFrame({
    'stenogramu_sk': [len(df)],
    'zodziu_sk': [df['zodziu_sk'].sum()],
    'vid_zodziu_stenogramoje': [df['zodziu_sk'].mean()],
    'med_zodziu_stenogramoje': [df['zodziu_sk'].median()],
    'sakiniu_sk': [df['sakiniu_sk'].sum()],
    'vid_sakinio_ilgis': [df['zodziu_sk'].sum() / df['sakiniu_sk'].sum()],
    'med_sakinio_ilgis': [np.median(visi_sakiniu_ilgiai) if visi_sakiniu_ilgiai else np.nan]
}, index=['Bendrai'])

galutine_lentele = pd.concat([kadenciju_lentele, bendra_eilute])

galutine_lentele = galutine_lentele.round(2)

print("\nPradinė stenogramų analizė")
print("-----------------------------")
print(galutine_lentele)

kadencijos = df['kadencija'].unique()

for kadencija in kadencijos:
    
    tekstas = " ".join(
        df[df['kadencija'] == kadencija]['tekstas'].astype(str)
    )

    wc = WordCloud(
        width=1600,
        height=800,
        background_color='white',
        collocations=False
    ).generate(tekstas)

    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Žodžių debesis: {kadencija}', fontsize=16)
    plt.show()

In [ ]:
def extract_speakers(text):
    if pd.isna(text):
        return []

    pattern = r'\b[A-ZĄČĘĖĮŠŲŪŽ]\.\s[A-ZĄČĘĖĮŠŲŪŽ]{2,}'
    return re.findall(pattern, str(text))

df['kalbetojai'] = df['tekstas'].apply(extract_speakers)

rezultatai = []

for kadencija in df['kadencija'].unique():

    subset = df[df['kadencija'] == kadencija]

    visi_kalbetojai = []

    for sarasas in subset['kalbetojai']:
        visi_kalbetojai.extend(sarasas)

    counter = Counter(visi_kalbetojai)

    unikalus_kiekis = len(counter)

    top5 = counter.most_common(5)

    top_rez = []

    for vardas, kiekis in top5:
        top_rez.append(f"{vardas} ({kiekis})")

    while len(top_rez) < 5:
        top_rez.append('')

    rezultatai.append({
        'kadencija': kadencija,
        'kalbėtojų skaičius': unikalus_kiekis,
        'TOP1': top_rez[0],
        'TOP2': top_rez[1],
        'TOP3': top_rez[2],
        'TOP4': top_rez[3],
        'TOP5': top_rez[4]
    })
    
    kalbetoju_lentele = pd.DataFrame(rezultatai)

print("\nKalbėtojų analizė pagal kadencijas")
print("-----------------------------------")
print(kalbetoju_lentele)

In [ ]:
time_pattern = r'(\d{1,2}\.\d{2}\s*val\.)'

columns_to_keep = [col for col in df.columns
                   if col not in ["tekstas", "klaida", "failo_pavadinimas", "failo_kelias"]]

rows = []

for idx, row in df.iterrows():
    text = str(row["tekstas"])
    parts = re.split(time_pattern, text)

    for i in range(1, len(parts), 2):
        laikas = parts[i].strip()
        segmentas = parts[i + 1].strip() if i + 1 < len(parts) else ""

        zodziu_kiekis = len(segmentas.split())

        if segmentas and zodziu_kiekis >= 30:
            new_row = {}

            for col in columns_to_keep:
                new_row[col] = row[col]

            new_row["laikas"] = laikas
            new_row["skyrelis"] = segmentas
            new_row["teksto_ilgis"] = len(segmentas)
            new_row["zodziu_kiekis"] = zodziu_kiekis

            rows.append(new_row)

df_segments = pd.DataFrame(rows)

print(df_segments.head())
print("Pradinis eilučių skaičius:", len(df))
print("Naujas skyrelių skaičius:", len(df_segments))

lt_stop = set(open("stopwords-lt.txt", encoding="utf-8").read().splitlines())
print(lt_stop)

In [ ]:
# Pašalina nereikšmingus žodžius iš teksto naudojant regex
def regex_clean(text):

    text = re.sub(r'\bNr\.\s*[A-Z0-9-]+\b', ' ', text, flags=re.IGNORECASE)

    patterns = [
        r"\b\d+\s*straipsn\w*\b",
        r"\b\d+\s*str\.\b",
        r"\b\d+\s*dalis\b",
        r"\b\d+\s*d\.\b",
        r"\bnr\b",
        r"\bm\b",
        r"\bd\b",
        r"\bstenograma\b",
        r"\b(lietuvos respublikos seim(as|o)|seim\w*)\b",
        r"\b(pavasario|rudens|žiemos|vasaros)\b",
        r"\b(ryt|vakar)\w*\b",
        r"\bposėdžio\b",
        r"\bsesij\w*\b",
        r"\blab(a|as) (diena|rytas|vakaras)\b",
        r"\b(dien(a|os)|ryt(as|o)|vakar(as|o))\b",
        r"\b(saus|vasar|kov|balandž|geguž|biržel|liep|rugpjūč|rugsė|spal|lapkr|gruod)\w*\b",
        r"\blietuvos respublikos\b",
        r"\bpirminink\w*\b",
        r"\bpavaduotoj\w*\b",
        r"\bkviečiu\b",
        r"\b(registr|užsireg)\w*\b",
        r"\b(prašau|prašome|prašom)\b",
        r"\bval\b",
        r"\bsėst\w*\b",
        r"\b(koleg\w*|gerbiam\w*)\b",
        r"\bužsireg\w*\b",
         r"\bačiū\b",
        r"\b(dėkoju|dėkui|dėkojame)\b",
        r"\bšiandien\w*\b",
        r"\b(pirmadien(io|is|į)|antradien(io|is)|trečiadien(io|is)|ketvirtadien(io|is)|penktadien(io|is)|šeštadien(io|is)|sekmadien(io|is))\b",
        r"\b(tiek|tam|tiem|tokiu|tokią|tokį|tokie|tokių|tas|tų|tuo|tokio|tokios|to|tos|tą|tie|todėl|tada|ten|tuos)\b",
        r"\b(šio|šių|šią|šį|šios|šioje|šiam|šito|šitą|šiuo|šiek)\b",
        r"\b(kitą|kitu|kitų|kiti|kuris|kurie|kurio|kuriuo|kuriuos|kuriam|kurių|kurios|kuria|kuri|kurį|kurią|ko|ką|kokio)\b",
        r"\b(jo|jos|juo|ja|jie|jos|jį|ją|jų|jūsų|jam|jai)\b",
        r"\b(viso|visi|visiems|visų|vieno|vieną)\b",
        r"\bįstatym\w*\b",
        r"\bprojekt\w*\b",
        r"\b(būti|būtų|bus|buvo|nebuvo|nebūtų|nebus|yra|nebėra|nėra)\b",
        r"\bpriimt\w*\b",
        r"\bpritart\w*\b",
        r"\bturim\w*\b",
        r"\bturi\b",
        r"\bčia\b",
        r"\blabai\b",
        r"\bgongas\b",
        r"\bjau\b",
        r"\blietuv\w*\b",
        r"\balsavo\b",
        r"\b(nariai|narys)\b",
        r"\bsusilaikė\b",
        r"\bpritarė\b",
        r"\bministr\w*\b",
        r"\bbalsavo\b",
        r"\bkomitet\w*\b",
        r"\bbalsuoj\w*\b",
        r"\bbalsuot\w*\b",
        r"\bmikrofon\w*\b",
        r"\bposėdį\b",
        r"\bpastab\w*\b",
        r"\bpasiūlym\w*\b",
        r"\bpritart\w*\b",
        r"\bnepritart\w*\b",
        r"\bsiūl\w*\b",
        r"\bbalsavimas pradėtas\b",
        r"\bsutar\w*\b",
        r"\bruošiasi\b",
        r"\bklausi\w*\b",
        r"\bgalim\w*\b",
        r"\bpriimame\b",
        r"\bpriėmim\w*\b",
        r"\bpastraipsniui\b",
        r"\bbendru\b",
        r"\bpritariam\w*\b",
        r"\bmotyv\w*\b",
        r"\bneužsiraš\w*\b",
        r"\bužsirašius\w*\b",
        r"\bniekas neužsiraš\w*\b",
        r"\bnorinčių kalbėti\b",
        r"\bkalbėt\w*\b",
        r"\bpaklaus\w*\b",
        r"\b(gauta|negauta|nėra gauta)\b",
        r"\bkiekvienam\b",
        r"\bbendru\b",
        r"\bnutarim\w*\b",
        r"\bsudarym\w*\b",
        r"\bpakeitim\w*\b",
        r"\bpapildym\w*\b",
        r"\bpatvirtinim\w*\b",
        r"\bpaskelbim\w*\b",
        r"\bpriedo\b",
        r"\bkodeks\w*\b",
        r"\bdarbotvark\w*\b",
        r"\brezervin\w*\b",
        r"\bataskait\w*\b",
        r"\brinkinio\b",
        r"\b(metų|metu|metai)\b",
        r"\batrodo\b",
        r"\bgalima\b",
        r"\breikia\b",
        r"\bkartu\b",
        r"\besmė\b",
        r"\baišku\b",
        r"\bmatyt\b",
        r"\bdalyk\w*\b",
        r"\bdabar\b",
        r"\bgerai\b",
        r"\bpavyzdžiui\b",
        r"\bkitaip\b",
        r"\bsvarstyti\b",
        r"\bžinote\b",
        r"\blaiko\b",
        r"\bkalbame\b",
        r"\bpranešėj\w*\b",
        r"\bviską\b",
        r"\b(nori|noriu|norėčiau)\b"
        r"\bnėra\b",
        r"\bstatut\w*\b",
        r"\bpremjer\w*\b",
        r"\bpriesaik\w*\b",
        r"\bprisiek\w*\b",
        r"\bmanote\b",
        r"\bapsispresk\w*\b",
        r"\bnepritar\w*\b",
        r"\bpritar\w*\b",
        r"\bbalsuo\w*\b",
        r"\bgalėt\w*\b",
        r"\btikrai\b",
        r"\bmanau\b",
        r"\bpatais\w*\b",
        r"\biš tikrųjų\b",
        r"\bsalė\w*\b",
        r"\bminėjau\b",
        r"\bdaug\w*\b",
        r"\bgali\b"
        ]

    for pattern in patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Pašalina skaičius ir visus simbolius, kurie nėra raidės, iš teksto
def basic_clean(text):
  text = re.sub(r"\d+", " ", text)
  text = re.sub(r"[^a-ząčęėįšųūž\s]", " ", text)
  return text

# Pašalina kalbėtojų kalbėtojų vardus iš teksto
def remove_speakers(text):

    text = re.sub(r"\b[A-ZĄČĘĖĮŠŲŪŽ]\.\s*(?:[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]+(?:-[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]+)*|[A-ZĄČĘĖĮŠŲŪŽ]+)\s*:?",
        " ", text)

    return text

# Sutvarko tarpus
def normalize_spaces(text):
  return re.sub(r"\s+", " ", text).strip()
  
# Pašalina nereikšmingus žodžius iš teksto
def stopwords_valymas(text, stopwords):
  return " ".join([w for w in text.split() if w not in stopwords])

# Apjungia visas valymo funkcijas
def clean(text, stopwords):
  text = str(text)
  text = re.sub(r"\(.*?\)", " ", text, flags=re.DOTALL) #valo tekstas skliaustuose
  text = remove_speakers(text)
  text = re.sub(r"\b[IVXLCDM]+\b", " ", text) # valo romėniškus skaičius
  text = text.lower()
  text = regex_clean(text)
  text = basic_clean(text)
  text = stopwords_valymas(text, stopwords)
  text = normalize_spaces(text)
  return text

df_2016_2020_segments = df_segments[df_segments["kadencija"] == "2016-2020"].copy()
df_2020_2024_segments = df_segments[df_segments["kadencija"] == "2020-2024"].copy()

df_2016_2020_segments["clean_text"] = df_2016_2020_segments["skyrelis"].apply(lambda x: clean(x, lt_stop))
df_2020_2024_segments["clean_text"] = df_2020_2024_segments["skyrelis"].apply(lambda x: clean(x, lt_stop))

sample_2016 = df_2016_2020_segments.sample(3, random_state=24)

print("\n===== 2016-2020 KADENCIJA =====\n")
for _, row in sample_2016.iterrows():
    print("--------- ORIGINALUS --------")
    print(row["skyrelis"][:5000])
    print("\n--------- VALYTAS ---------")
    print(row["clean_text"][:5000])
    print("\n====================\n")

sample_2016 = df_2020_2024_segments.sample(3, random_state=24)

print("\n===== 2020-2024 KADENCIJA =====\n")
for _, row in sample_2016.iterrows():
    print("--------- ORIGINALUS --------")
    print(row["skyrelis"][:5000])
    print("\n--------- VALYTAS ---------")
    print(row["clean_text"][:5000])
    print("\n====================\n")

In [ ]:
def generate_wordcloud(df_kad, title=None):
    tekstas = " ".join(df_kad['clean_text'].astype(str))
    wc = WordCloud(width=1600, height=800, background_color='white', collocations=False).generate(tekstas)
    plt.figure(figsize=(14,7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    if title is None:
        title = df_kad['kadencija'].iloc[0]
    plt.title(f'Žodžių debesis: {title}', fontsize=16)
    plt.show()

generate_wordcloud(df_2016_2020_segments, title="2016-2020")
generate_wordcloud(df_2020_2024_segments, title="2020-2024")

In [ ]:
MODEL_NAME = "EMBEDDIA/litlat-bert"

tokenizer_path = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json")
tokenizer = Tokenizer.from_file(tokenizer_path)

model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

labels = {
    "gynyba_ir_uzsienis": (
        "krašto apsauga kariuomenė nato nacionalinis saugumas gynyba karas "
        "ukraina rusija sienų apsauga užsienio politika diplomatija sankcijos "
        "tarptautiniai santykiai geopolitika misijos"
    ),

    "ekonomika_ir_mokesciai": (
        "ekonomika biudžetas finansai mokesčiai pvm pajamos išlaidos deficitas "
        "infliacija investicijos ekonomikos augimas fiskalinė politika fondai"
    ),

    "socialine_politika": (
        "socialinė apsauga sveikata švietimas ligoninės gydytojai mokyklos "
        "pensijos pašalpos šeimos politika vaiko teisės gerovė neįgalieji"
    ),

    "teise_ir_teisetvarka": (
        "teisė teisėtvarka teismai policija prokuratūra konstitucinis teismas "
        "baudžiamoji atsakomybė administraciniai nusižengimai įstatymai"
    ),

    "vidaus_politika_ir_institucijos": (
        "vidaus politika seimas vyriausybė institucijos rinkimai partijos vrk "
        "komisijos įstatymų leidyba valdymas politinė sistema"
    ),

    "aplinka_ir_zemes_ukis": (
        "aplinkos apsauga klimato kaita ekologija miškai vanduo atliekos "
        "žemės ūkis ūkininkai kaimas žuvininkystė gamtos ištekliai"
    ),

    "energetika_ir_infrastruktura": (
        "energetika elektros energija dujos šildymas kainos transportas "
        "infrastruktūra viešieji pirkimai atsinaujinanti energija"
    ),

    "kultura_ir_istorija": (
        "kultūra menas paveldas muziejai istorija istorinė atmintis genocidas "
        "rezistencija švietimo kultūra mokslas universitetai"
    )
}

label_names = list(labels.keys())
label_texts = list(labels.values())

In [ ]:
def encode_batch(texts, max_length=512):
    input_ids_list = []
    attention_masks = []

    for text in texts:
        enc = tokenizer.encode(str(text))
        ids = enc.ids[:max_length]
        attn = [1] * len(ids)
        if len(ids) < max_length:
            pad_len = max_length - len(ids)
            ids = ids + [1] * pad_len
            attn = attn + [0] * pad_len
        input_ids_list.append(ids)
        attention_masks.append(attn)

    input_ids = torch.tensor(input_ids_list, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_masks, dtype=torch.long).to(device)
    return input_ids, attention_mask

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    counted = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return summed / counted

def get_embeddings(texts, batch_size=8, max_length=512):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Skaičiuojami embeddingai"):
        batch_texts = texts[i:i + batch_size]
        input_ids, attention_mask = encode_batch(batch_texts, max_length=max_length)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_embeddings = mean_pooling(outputs, attention_mask)
        batch_embeddings = batch_embeddings.cpu().numpy()
        all_embeddings.append(batch_embeddings)
    return np.vstack(all_embeddings)

label_embeddings = get_embeddings(label_texts, batch_size=4)

In [ ]:
def classification(texts, label_names, label_embeddings, batch_size=8):
    text_embeddings = get_embeddings(texts, batch_size=batch_size)
    similarity_matrix = cosine_similarity(text_embeddings, label_embeddings)
    best_idx = similarity_matrix.argmax(axis=1)
    predicted_labels = [label_names[i] for i in best_idx]
    predicted_scores = similarity_matrix.max(axis=1)
    return predicted_labels, predicted_scores, similarity_matrix

df_work = pd.concat([df_2016_2020_segments, df_2020_2024_segments], ignore_index=True)
text_column = "clean_text"

predicted_labels, predicted_scores, similarity_matrix = classification(
    texts=df_work[text_column].tolist(),
    label_names=label_names,
    label_embeddings=label_embeddings,
    batch_size=8
)

df_work["tema"] = predicted_labels
df_work["temos_patikimumas"] = predicted_scores

for i, tema in enumerate(label_names):
    df_work[f"sim_{tema}"] = similarity_matrix[:, i]

df_temos = df_work.copy()

print("\n=== Pirmi klasifikavimo rezultatai ===")
display(df_temos[["kadencija", "laikas", "skyrelis", text_column, "tema", "temos_patikimumas"]].head(10))

print("\n=== Temų bendras pasiskirstymas ===")
tema_counts = df_temos["tema"].value_counts()
display(tema_counts.to_frame(name="kiekis"))

print("\n=== Temų pasiskirstymas pagal kadenciją ===")
tema_kad = pd.crosstab(df_temos["kadencija"], df_temos["tema"])
display(tema_kad)

tema_kad.plot(kind="bar", figsize=(12, 6))

plt.title("Temų pasiskirstymas pagal kadencijas")
plt.xlabel("Kadencija")
plt.ylabel("Įrašų skaičius")
plt.xticks(rotation=0)
plt.legend(
    title="Tema",
    labels=[
        "Aplinka ir žemės ūkis",
        "Ekonomika ir finansai",
        "Energetika ir infrastruktūra",
        "Gynyba ir užsienio politika",
        "Kultūra ir istorija",
        "Socialinė politika",
        "Teisė ir teisėtvarka",
        "Vidaus politika ir institucijos"])
plt.tight_layout()
plt.show()

In [ ]:
def optimal_k(X, k_range=range(2, 30), label=None):
    sil_scores = []

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=8, n_init=10)
        labels = kmeans.fit_predict(X)

        score = silhouette_score(X, labels, metric='cosine')
        sil_scores.append(score)

    plt.plot(list(k_range), sil_scores, marker='o', label=label)
    plt.title("Silhouette metodo įvertis pagal klasterių skaičių")
    plt.xlabel("Klasterių skaičius (k)")
    plt.ylabel("Silhouette įvertis")
    plt.legend()

    best_k = list(k_range)[np.argmax(sil_scores)]
    print("Optimalus klasterių sk.:", best_k)

    return best_k

def cluster_text(df_kad, n_clusters=None):

    vectorizer = TfidfVectorizer(
        max_df=0.80,
        min_df=5,
        max_features=20000,
        ngram_range=(1, 2)
    )

    X = vectorizer.fit_transform(df_kad['clean_text'])

    if n_clusters is None:
      n_clusters = optimal_k(X, label=df_kad['kadencija'].iloc[0])

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=8,
        n_init=10,
        init='k-means++',
        max_iter=300
    )

    kmeans.fit(X)

    df_kad = df_kad.copy()
    df_kad['cluster'] = kmeans.labels_

    terms = vectorizer.get_feature_names_out()
    centroids = kmeans.cluster_centers_

    print(f"\n--- {df_kad['kadencija'].iloc[0]} klasteriai ---")

    for i in range(n_clusters):
        top_terms = centroids[i].argsort()[-15:][::-1]
        print(f"{i} klasteris: {', '.join(terms[j] for j in top_terms)}")

    return df_kad, kmeans, vectorizer, X

In [ ]:
# Artimiausių tekstų ištraukimas
def closest_texts(df, X, kmeans, cluster_id, n_return=5):
    df = df.reset_index(drop=True)

    mask = (df['cluster'].values == cluster_id)
    X_cluster = X[mask]

    if X_cluster.shape[0] == 0:
        return []

    centroid = kmeans.cluster_centers_[cluster_id].reshape(1, -1)

    distances = pairwise_distances(X_cluster, centroid, metric='cosine').ravel()

    idx_sorted = np.argsort(distances)

    return df.loc[mask, 'clean_text'].iloc[idx_sorted[:n_return]].tolist()

# 2016–2020 klasterizavimas
df_2016_2020_segments, kmeans_2016, vect_2016, X_2016 = cluster_text(df_2016_2020_segments)

# 2020–2024 klasterizavimas
df_2020_2024_segments, kmeans_2020, vect_2020, X_2020 = cluster_text(df_2020_2024_segments)

# Tekstų skaičius kiekviename klasteryje
print(df_2016_2020_segments['cluster'].value_counts().sort_index())
print(df_2020_2024_segments['cluster'].value_counts().sort_index())

# gynyba ir užsienio politika 2016
texts_2016_c23 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 23, 140)
# gynyba ir užsienio politika 2020
texts_2020_27 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 27, 140)

# ekonomika ir mokesčiai 2016
texts_2016_c3 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 3, 94)
# ekonomika ir mokesčiai  2020
texts_2020_c8 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 8, 93)
texts_2020_c21 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 21, 93)

# socialinė politika 2016
texts_2016_c1 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 1, 24)
texts_2016_c2 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 2, 24)
texts_2016_c4 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 4, 23)
texts_2016_c6 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 6, 23)
texts_2016_c18 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 18, 23)
texts_2016_c19 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 19, 23)
# socialinė politika 2020
texts_2020_c1 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 1, 24)
texts_2020_c12 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 12, 24)
texts_2020_c13 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 13, 23)
texts_2020_c16 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 16, 23)
texts_2020_c18 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 18, 23)
texts_2020_c26 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 26, 23)

In [ ]:
# teisė ir teisėtvarka 2016
texts_2016_c7 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 7, 17)
texts_2016_c10 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 10, 17)
texts_2016_c11 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 11, 17)
texts_2016_c15 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 15, 17)
texts_2016_c16 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 16, 16)
texts_2016_c17 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 17, 16)
texts_2016_c20 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 20, 16)
texts_2016_c21 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 21, 16)
texts_2016_c24 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 24, 16)
# teisė ir teisėtvarka 2020
texts_2020_c2 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 2, 17)
texts_2020_c3 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 3, 17)
texts_2020_c5 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 5, 17)
texts_2020_c7 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 7, 17)
texts_2020_c14 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 14, 16)
texts_2020_c15 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 15, 16)
texts_2020_c17 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 17, 16)
texts_2020_c23 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 23, 16)

# vidus politika ir institucijos 2016
texts_2016_c0 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 0, 20)
texts_2016_c5 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 5, 20)
texts_2016_c9 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 9, 20)
texts_2016_c12 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 12, 20)
texts_2016_c26 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 26, 20)
texts_2016_c27 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 27, 20)
texts_2016_c28 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 28, 20)
# vidus politika ir institucijos 2020
texts_2020_c0 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 0, 20)
texts_2020_c6 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 6, 20)
texts_2020_c9 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 9, 20)
texts_2020_c10 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 10, 20)
texts_2020_c20 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 20, 20)
texts_2020_c24 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 24, 20)
texts_2020_c28 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 28, 20)

# aplinka ir žemės ūkis 2016
texts_2016_c22 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 22, 140)
# aplinka ir žemės ūkis 2020
texts_2020_c11 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 11, 140)

In [ ]:
# energetika ir infrastruktūra 2016
texts_2016_c14 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 14, 140)
# energetika ir infrastruktūra 2020
texts_2020_c4 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 4, 140)

# kultūra ir istorija 2016
texts_2016_c13 = closest_texts(df_2016_2020_segments, X_2016, kmeans_2016, 13, 106)
# kultūra ir istorija 2020
texts_2020_c19 = closest_texts(df_2020_2024_segments, X_2020, kmeans_2020, 19, 150)

In [ ]:
# Testavimo aibės sukūrimas
test_df_2016 = df_2016_2020_segments[
    df_2016_2020_segments['cluster'].isin([8, 25])
]

test_df_2020 = df_2020_2024_segments[
    df_2020_2024_segments['cluster'].isin([22, 25])
]

test_df = pd.concat([test_df_2016, test_df_2020], ignore_index=True)

print(test_df.shape)

test_df.to_parquet("test_df.parquet", index=False)

# Duomenų žymėjimas
def labeling(cluster_texts_dict):

    data = []

    for label, texts in cluster_texts_dict.items():
        for t in texts:
            data.append({"text": t, "label": label})

    return pd.DataFrame(data)

cluster_texts = {
    "gynyba_ir_uzsienis": texts_2016_c23 + texts_2020_27,
    "ekonomika_ir_mokesciai": texts_2016_c3 + texts_2020_c8 + texts_2020_c21,
    "socialine_politika": texts_2016_c1 + texts_2016_c2 + texts_2016_c4 + texts_2016_c6 + texts_2016_c18 + texts_2016_c19 + texts_2020_c1 + texts_2020_c12 + texts_2020_c13 + texts_2020_c16 + texts_2020_c18 + texts_2020_c26,
    "teise_ir_teisetvarka": texts_2016_c7 + texts_2016_c10 + texts_2016_c11 + texts_2016_c15 + texts_2016_c16 + texts_2016_c17 + texts_2016_c20 + texts_2016_c21 + texts_2016_c24 + texts_2020_c2 + texts_2020_c3 + texts_2020_c5 + texts_2020_c7 + texts_2020_c14 + texts_2020_c15 + texts_2020_c17 + texts_2020_c23,
    "vidaus_politika_ir_institucijos": texts_2016_c0 + texts_2016_c5 + texts_2016_c9 + texts_2016_c12 + texts_2016_c26 + texts_2016_c27 + texts_2016_c28 + texts_2020_c0 + texts_2020_c6 + texts_2020_c9 + texts_2020_c10 + texts_2020_c20 + texts_2020_c24 + texts_2020_c28,
    "aplinka_ir_zemes_ukis": texts_2016_c22 + texts_2020_c11,
    "energetika_ir_infrastruktura": texts_2016_c14 + texts_2020_c4,
    "kultura_ir_istorija": texts_2016_c13 + texts_2020_c19
}

tuning_df = labeling(cluster_texts).sample(frac=1, random_state=8).reset_index(drop=True)
tuning_df.to_parquet("tuning_dataset.parquet", index=False)

total = sum(len(texts) for texts in cluster_texts.values())
print(total)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df, val_df = train_test_split(tuning_df, test_size=0.1, stratify=tuning_df["label"], random_state=8)

labels = sorted(tuning_df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)

In [ ]:
MODEL_NAME = "EMBEDDIA/litlat-bert"

tokenizer_path = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json")
tokenizer = Tokenizer.from_file(tokenizer_path)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.gradient_checkpointing_enable()
model.to(device)

class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()

        self.pad_id = tokenizer.token_to_id("<pad>")
        self.cls_id = tokenizer.token_to_id("<s>")
        self.sep_id = tokenizer.token_to_id("</s>")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = tokenizer.encode(self.texts[idx])

        input_ids = [self.cls_id] + encoding.ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)

        max_len = 256

        if len(input_ids) > max_len:
            input_ids = input_ids[:max_len]
            attention_mask = attention_mask[:max_len]
        else:
            pad_len = max_len - len(input_ids)
            input_ids += [self.pad_id] * pad_len
            attention_mask += [0] * pad_len

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(self.labels[idx])
        }

train_dataset = TextDataset(train_df)
val_dataset = TextDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    num_workers=2,
    pin_memory=True
)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

epochs = 3
num_training_steps = epochs * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

scaler = GradScaler()

start_epoch = 0

for epoch in range(start_epoch, epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        with autocast():
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train loss: {avg_train_loss:.4f}")

    model.eval()

    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            with autocast():
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"Val loss: {avg_val_loss:.4f}")
    print(f"Val accuracy: {acc:.4f}")
    print(f"Val F1: {f1:.4f}")

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": lr_scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_acc": acc,
        "val_f1": f1,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    torch.save(checkpoint, f"checkpoint_epoch_{epoch+1}.pt")

    print(f"Checkpoint saved: epoch_{epoch+1}")

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, df):
        self.texts = df["clean_text"].tolist()

        self.pad_id = tokenizer.token_to_id("<pad>")
        self.cls_id = tokenizer.token_to_id("<s>")
        self.sep_id = tokenizer.token_to_id("</s>")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer.encode(self.texts[idx])

        input_ids = [self.cls_id] + encoding.ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)

        max_len = 256

        if len(input_ids) > max_len:
            input_ids = input_ids[:max_len]
            attention_mask = attention_mask[:max_len]
        else:
            pad_len = max_len - len(input_ids)
            input_ids += [self.pad_id] * pad_len
            attention_mask += [0] * pad_len

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask)
        }

def predict(model, loader):
    model.eval()
    preds_all = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            with autocast():
                logits = model(**batch).logits

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())

    return preds_all

test_dataset = InferenceDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=8)

test_preds = predict(model, test_loader)
test_df["pred_label"] = [id2label[i] for i in test_preds]

test_df[["clean_text", "pred_label"]].sample(10, random_state=42)

label_mapping = {
    "aplinka_ir_zemes_ukis": "Aplinkos ir žemės ūkis",
    "ekonomika_ir_mokesciai": "Ekonomika ir mokesčiai",
    "energetika_ir_infrastruktura": "Energetika ir infrastruktūra",
    "gynyba_ir_uzsienis": "Gynyba ir užsienis",
    "kultura_ir_istorija": "Kultūra ir istorija",
    "socialine_politika": "Socialinė politika",
    "teise_ir_teisetvarka": "Teisė ir teisėtvarka",
    "vidaus_politika_ir_institucijos": "Vidaus politika ir institucijos"
}

label_counts = test_df["pred_label"].value_counts()

label_counts.index = [
    label_mapping[label]
    for label in label_counts.index
]

plt.figure(figsize=(10, 8))

ax = label_counts.sort_values().plot(
    kind="barh",
    color="#4C72B0"
)

plt.title("Mišrių klasterių prognozuotų klasių pasiskirstymas")
plt.xlabel("Priskirtų tekstų skaičius")
plt.ylabel("Prognozuota klasė")

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visų tekstų klasifikavimas
inf_dataset_1 = InferenceDataset(df_2016_2020_segments)
inf_loader_1 = DataLoader(inf_dataset_1, batch_size=8)

inf_dataset_2 = InferenceDataset(df_2020_2024_segments)
inf_loader_2 = DataLoader(inf_dataset_2, batch_size=8)

preds_2016_2020 = predict(model, inf_loader_1)
preds_2020_2024 = predict(model, inf_loader_2)

df_2016_2020_segments["pred_label"] = [id2label[i] for i in preds_2016_2020]
df_2020_2024_segments["pred_label"] = [id2label[i] for i in preds_2020_2024]

sampled_texts = (
    df_2016_2020_segments
    .groupby("pred_label")["skyrelis"]
    .apply(lambda x: x.sample(min(4, len(x)), random_state=8))
)

for label in sampled_texts.index.get_level_values(0).unique():
    print("\n" + "="*80)
    print(label)
    print("="*80)

    texts = sampled_texts.loc[label]

    for i, text in enumerate(texts, 1):
        print(f"\n{i}. {text}\n")

counts_2016_2020 = df_2016_2020_segments["pred_label"].value_counts().sort_index()
counts_2020_2024 = df_2020_2024_segments["pred_label"].value_counts().sort_index()

comparison_df = pd.DataFrame({
    "2016–2020": counts_2016_2020,
    "2020–2024": counts_2020_2024
}).fillna(0)

print(comparison_df)

new_label = [
    "Aplinkos ir žemės ūkis",
    "Ekonomika ir finansai",
    "Energetika ir infrastruktūra",
    "Gynyba ir užsienio politika",
    "Kultūra ir istorija",
    "Socialinė politika",
    "Teisė ir teisetvarka",
    "Vidaus politika ir institucijos"
]

ax = comparison_df.plot(
    kind="bar",
    figsize=(12, 7),
    color=["#80668c", "#709470"]
)

plt.title("Prognozuotų klasių pasiskirstymas", fontsize=16)
plt.xlabel("Prognozuota klasė", fontsize=13)
plt.ylabel("Tekstų skaičius", fontsize=13)

ax.set_xticklabels(new_label, rotation=30, ha='right')

plt.legend(title="Laikotarpis")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%d',
        padding=3,
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Svarbiausių frazių išrinkimas naudojant TF-IDF metodą
df1 = pd.read_parquet("pred_2016_2020.parquet")
df2 = pd.read_parquet("pred_2020_2024.parquet")

text_col = "clean_text"
top_n = 3

def extract_tfidf_phrases(df, df_name, kadencija):
    results = []

    labels = df["pred_label"].dropna().unique()

    # Progress bar per temas
    for label in tqdm(labels, desc=f"Processing {df_name}"):

        texts = (
            df.loc[df["pred_label"] == label, text_col]
            .dropna()
            .astype(str)
        )

        vectorizer = TfidfVectorizer(
            ngram_range=(2, 3),
            sublinear_tf=False
        )

        tfidf_matrix = vectorizer.fit_transform(texts)

        feature_names = vectorizer.get_feature_names_out()

        mean_tfidf = tfidf_matrix.mean(axis=0).A1

        top_indices = mean_tfidf.argsort()[::-1][:top_n]

        for rank, idx in enumerate(top_indices, start=1):
            results.append({
                "dataset": df_name,
                "kadencija": kadencija,
                "tema": label,
                "rank": rank,
                "fraze": feature_names[idx],
                "tfidf_score": round(mean_tfidf[idx], 5)
            })

    return pd.DataFrame(results)

df1_tfidf_phrases = extract_tfidf_phrases(
    df1,
    "df1",
    "2016-2020"
)

df2_tfidf_phrases = extract_tfidf_phrases(
    df2,
    "df2",
    "2020-2024"
)

df1_tfidf_phrases.to_parquet(
    "2016-2020_tfidf_phrases.parquet",
    index=False
)

df2_tfidf_phrases.to_parquet(
    "2020-2024_tfidf_phrases.parquet",
    index=False
)

print("Parquet failai išsaugoti.")

In [ ]:
# Svarbiausių frazių išrinkimas naudojant YAKE metodą

df1 = pd.read_parquet("pred_2016_2020.parquet")
df2 = pd.read_parquet("pred_2020_2024.parquet")

lt_stop = set(open("stopwords-lt.txt", encoding="utf-8").read().splitlines())
print(lt_stop)

REGEX_PATTERNS = [
    r"\b\d+\s*straipsn\w*\b",
    r"\b\d+\s*str\.\b",
    r"\b\d+\s*dalis\b",
    r"\b\d+\s*d\.\b",
    r"\bnr\b",
    r"\bm\b",
    r"\bd\b",
    r"\bstenograma\b",
    r"\b(lietuvos respublikos seim(as|o)|seim\w*)\b",
    r"\b(pavasario|rudens|žiemos|vasaros)\b",
    r"\b(ryt|vakar)\w*\b",
    r"\bposėdžio\b",
    r"\bsesij\w*\b",
    r"\blab(a|as) (diena|rytas|vakaras)\b",
    r"\b(dien(a|os)|ryt(as|o)|vakar(as|o))\b",
    r"\b(saus|vasar|kov|balandž|geguž|biržel|liep|rugpjūč|rugsė|spal|lapkr|gruod)\w*\b",
    r"\blietuvos respublikos\b",
    r"\bpirminink\w*\b",
    r"\bpavaduotoj\w*\b",
    r"\bkviečiu\b",
    r"\b(registr|užsireg)\w*\b",
    r"\b(prašau|prašome|prašom)\b",
    r"\bval\b",
    r"\bsėst\w*\b",
    r"\b(koleg\w*|gerbiam\w*)\b",
    r"\bužsireg\w*\b",
    r"\bačiū\b",
    r"\b(dėkoju|dėkui|dėkojame)\b",
    r"\bšiandien\w*\b",
    r"\b(pirmadien(io|is|į)|antradien(io|is)|trečiadien(io|is)|ketvirtadien(io|is)|penktadien(io|is)|šeštadien(io|is)|sekmadien(io|is))\b",
    r"\b(tiek|tam|tiem|tokiu|tokią|tokį|tokie|tokių|tas|tų|tuo|tokio|tokios|to|tos|tą|tie|todėl|tada|ten|tuos)\b",
    r"\b(šio|šių|šią|šį|šios|šioje|šiam|šito|šitą|šiuo|šiek)\b",
    r"\b(kitą|kitu|kitų|kiti|kuris|kurie|kurio|kuriuo|kuriuos|kuriam|kurių|kurios|kuria|kuri|kurį|kurią|ko|ką|kokio)\b",
    r"\b(jo|jos|juo|ja|jie|jos|jį|ją|jų|jūsų|jam|jai)\b",
    r"\b(viso|visi|visiems|visų|vieno|vieną)\b",
    r"\bįstatym\w*\b",
    r"\bprojekt\w*\b",
    r"\b(būti|būtų|bus|buvo|nebuvo|nebūtų|nebus|yra|nebėra|nėra)\b",
    r"\bpriimt\w*\b",
    r"\bpritart\w*\b",
    r"\bturim\w*\b",
    r"\bturi\b",
    r"\bčia\b",
    r"\blabai\b",
    r"\bgongas\b",
    r"\bjau\b",
    r"\blietuv\w*\b",
    r"\balsavo\b",
    r"\b(nariai|narys)\b",
    r"\bsusilaikė\b",
    r"\bpritarė\b",
    r"\bministr\w*\b",
    r"\bkomitet\w*\b",
    r"\bbalsuoj\w*\b",
    r"\bbalsuot\w*\b",
    r"\bmikrofon\w*\b",
    r"\bposėdį\b",
    r"\btvirtin\w*\b",
    r"\bpastab\w*\b",
    r"\bpasiūlym\w*\b",
    r"\bnepritart\w*\b",
    r"\bsiūl\w*\b",
    r"\bbalsavimas pradėtas\b",
    r"\bsutar\w*\b",
    r"\bruošiasi\b",
    r"\bklausi\w*\b",
    r"\bgalim\w*\b",
    r"\bpriimame\b",
    r"\bstraipsn\w*\b",
    r"\bpriėmim\w*\b",
    r"\bpastraipsniui\b",
    r"\bbendru\b",
    r"\bpritariam\w*\b",
    r"\bmotyv\w*\b",
    r"\bneužsiraš\w*\b",
    r"\bužsirašius\w*\b",
    r"\bniekas neužsiraš\w*\b",
    r"\bnorinčių kalbėti\b",
    r"\bkalbėt\w*\b",
    r"\bpaklaus\w*\b",
    r"\b(gauta|negauta|nėra gauta)\b",
    r"\bkiekvienam\b",
    r"\bnutarim\w*\b",
    r"\bsudarym\w*\b",
    r"\bpakeitim\w*\b",
    r"\bpapildym\w*\b",
    r"\bpatvirtinim\w*\b",
    r"\bpaskelbim\w*\b",
    r"\bpriedo\b",
    r"\bkodeks\w*\b",
    r"\bdarbotvark\w*\b",
    r"\brezervin\w*\b",
    r"\bataskait\w*\b",
    r"\brinkinio\b",
    r"\b(metų|metu|metai)\b",
    r"\batrodo\b",
    r"\bgalima\b",
    r"\breikia\b",
    r"\bkartu\b",
    r"\besmė\b",
    r"\baišku\b",
    r"\bmatyt\b",
    r"\bdalyk\w*\b",
    r"\bdabar\b",
    r"\bgerai\b",
    r"\bpavyzdžiui\b",
    r"\bkitaip\b",
    r"\bsvarstyti\b",
    r"\bžinote\b",
    r"\blaiko\b",
    r"\bkalbame\b",
    r"\bpranešėj\w*\b",
    r"\bviską\b",
    r"\b(nori|noriu|norėčiau)\b",
    r"\bvienbals\w*\b",
    r"\bsusilaik\w*\b",
    r"\bdien\w*\b",
    r"\bvaland\w*\b",
    r"\bpatiksl\w*\b",
    r"\btikslin\w*\b",
    r"\bposėd\w*\b",
    r"\bvardu\b",
    r"\bstatut\w*\b",
    r"\bpremjer\w*\b",
    r"\bpriesaik\w*\b",
    r"\bprisiek\w*\b",
    r"\bsveiki\b",
    r"\bmanote\b",
    r"\bapsispresk\w*\b",
    r"\bbalsuo\w*\b",
    r"\bgalėt\w*\b",
    r"\btikrai\b",
    r"\bmanau\b",
    r"\bpatais\w*\b",
    r"\biš tikrųjų\b",
    r"\bsalė\w*\b",
    r"\bminėjau\b",
    r"\bdaug\w*\b",
    r"\bgali\b"
]

COMBINED_REGEX = re.compile("|".join(REGEX_PATTERNS), flags=re.IGNORECASE)

NR_REGEX = re.compile(r"\bNr\.\s*[A-Z0-9-]+\b", flags=re.IGNORECASE)
BRACKETS_REGEX = re.compile(r"\(.*?\)", flags=re.DOTALL)
ROMAN_REGEX = re.compile(r"\b[IVXLCDM]+\b", flags=re.IGNORECASE)
DIGITS_REGEX = re.compile(r"\d+")
BASIC_ALLOWED_REGEX = re.compile(
    r"[^a-ząčęėįšųūž\.\,\!\?\:\;\–\(\)\s]",
    flags=re.IGNORECASE
)
SPACES_REGEX = re.compile(r"\s+")

SPEAKER_REGEX = re.compile(
    r"\b[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž\-]+\.?:?"
)

NAME_PAIR = r"[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž]+){1,2}"
NAME_LIST_REGEX = re.compile(rf"(?:{NAME_PAIR}\s+){{4,}}")

In [ ]:
def remove_speakers_and_name_lists(text):
    text = SPEAKER_REGEX.sub(" ", text)
    text = NAME_LIST_REGEX.sub(" ", text)
    return text


def regex_clean(text):
    text = NR_REGEX.sub(" ", text)
    text = COMBINED_REGEX.sub(" ", text)
    return SPACES_REGEX.sub(" ", text).strip()


def basic_clean(text):
    text = DIGITS_REGEX.sub(" ", text)
    text = BASIC_ALLOWED_REGEX.sub(" ", text)
    return text


def clean_punctuation_artifacts(text):
    text = re.sub(r"\s+[–-]\s*[.,!?;:]\s+", " ", text)
    text = re.sub(r"[.,!?;:]\s+[–-]\s*[.,!?;:]", " ", text)
    text = re.sub(r"\.\s+–\s+", ". ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"\s*–\s*", " – ", text)
    text = re.sub(r"([.,!?;:]){2,}", r"\1", text)
    return SPACES_REGEX.sub(" ", text).strip()


def stopwords_valymas(text, stopwords):
    return " ".join(
        w for w in text.split()
        if w not in stopwords
    )


def clean(text, stopwords):
    text = str(text)

    text = BRACKETS_REGEX.sub(" ", text)
    text = remove_speakers_and_name_lists(text)
    text = ROMAN_REGEX.sub(" ", text)
    text = text.lower()
    text = regex_clean(text)
    text = basic_clean(text)
    text = clean_punctuation_artifacts(text)
    text = stopwords_valymas(text, stopwords)
    text = SPACES_REGEX.sub(" ", text).strip()

    return text

df_2016_2020_segments = df1[df1["kadencija"] == "2016-2020"].copy()
df_2020_2024_segments = df2[df2["kadencija"] == "2020-2024"].copy()

df_2016_2020_segments["clean_text"] = df_2016_2020_segments["skyrelis"].apply(lambda x: clean(x, lt_stop))
df_2020_2024_segments["clean_text"] = df_2020_2024_segments["skyrelis"].apply(lambda x: clean(x, lt_stop))

text_col = "clean_text"
top_n = 10

In [ ]:
def has_repeated_word(phrase):
    words = phrase.lower().split()
    return len(words) != len(set(words))

def extract_yake_phrases(df, df_name, kadencija):
    results = []

    labels = df["pred_label"].dropna().unique()

    for label in tqdm(labels, desc=f"Processing {df_name}"):

        texts = (
            df.loc[df["pred_label"] == label, text_col]
            .dropna()
            .astype(str)
        )

        # Sujungia visus temos tekstus
        combined_text = " ".join(texts)

        kw_extractor = yake.KeywordExtractor(
            lan="lt",
            n=3,
            top=top_n * 3,
            dedupLim=0.5,
            stopwords=None
        )

        keywords = kw_extractor.extract_keywords(combined_text)

        filtered_keywords = []

        for phrase, score in keywords:
            if not has_repeated_word(phrase):
                filtered_keywords.append((phrase, score))

            if len(filtered_keywords) == top_n:
                break

        for rank, (phrase, score) in enumerate(filtered_keywords, start=1):
            results.append({
                "dataset": df_name,
                "kadencija": kadencija,
                "tema": label,
                "rank": rank,
                "fraze": phrase,
                "yake_score": score
            })

    return pd.DataFrame(results)

df1_yake_phrases = extract_yake_phrases(
    df1,
    "df1",
    "2016-2020"
)

df2_yake_phrases = extract_yake_phrases(
    df2,
    "df2",
    "2020-2024"
)

df1_yake_phrases.to_parquet(
    "2016-2020_yake_phrases_sakiniai3.parquet",
    index=False
)

df2_yake_phrases.to_parquet(
    "2020-2024_yake_phrases_sakiniai3.parquet",
    index=False
)

print("YAKE parquet failai išsaugoti.")

In [ ]:
# Svarbiausių frazių išrinkimas naudojant KeyBERT metodą

df1 = pd.read_parquet("pred_2016_2020.parquet")
df2 = pd.read_parquet("pred_2020_2024.parquet")

REGEX_PATTERNS = [
    # Teisėkūros / dokumentų technika
    r"\b\d+\s*straipsn\w*\b",
    r"\b\d+\s*str\.\b",
    r"\b\d+\s*dal\w*\b",
    r"\b\d+\s*d\.\b",
    r"\bnr\.?\b",
    r"\bstenogram\w*\b",
    r"\bįstatym\w*\b",
    r"\bprojekt\w*\b",
    r"\bnutarim\w*\b",
    r"\bkodeks\w*\b",
    r"\bdarbotvark\w*\b",
    r"\bpakeitim\w*\b",
    r"\bpapildym\w*\b",
    r"\bpatvirtinim\w*\b",
    r"\bpaskelbim\w*\b",
    r"\bpried\w*\b",
    r"\brinkin\w*\b",
    r"\bdepartament\w*\b",
    r"\bministerij\w*\b",
    r"\bnuostat\w*\b",
    r"\bprocedūr\w*\b",
    r"\bpunkt\w*\b",
    r"\bstraipsn\w*\b",
    r"\bpatais\w*\b",
    r"\bstatut\w*\b",

    # Seimo / posėdžio procedūros
    r"\b(lietuvos respublikos seim(as|o)|seim\w*)\b",
    r"\blietuvos respublikos\b",
    r"\bposėd\w*\b",
    r"\bsesij\w*\b",
    r"\bpirminink\w*\b",
    r"\bpavaduotoj\w*\b",
    r"\bkomitet\w*\b",
    r"\bministr\w*\b",
    r"\bpranešėj\w*\b",
    r"\btribūn\w*\b",
    r"\bmikrofon\w*\b",
    r"\bsal\w*\b",
    r"\bgong\w*\b",

    # Procedūriniai veiksmai
    r"\bkvie\w*\b",
    r"\bpraš\w*\b",
    r"\bregistr\w*\b",
    r"\bužsireg\w*\b",
    r"\bsėst\w*\b",
    r"\bpriimt\w*\b",
    r"\bpriim\w*\b",
    r"\bpriėm\w*\b",
    r"\bpriėmim\w*\b",
    r"\bprit\w*\b",
    r"\bpritar\w*\b",
    r"\bneprit\w*\b",
    r"\bnepritar\w*\b",
    r"\bbals\w*\b",
    r"\bsusilaik\w*\b",
    r"\bsiūl\w*\b",
    r"\bsutar\w*\b",
    r"\bklaus\w*\b",
    r"\bpaklaus\w*\b",
    r"\bkalbėt\w*\b",
    r"\bužsiraš\w*\b",
    r"\bneužsiraš\w*\b",
    r"\bpastraipsniui\b",
    r"\bbendru\b",
    r"\bmotyv\w*\b",
    r"\bpasiūlym\w*\b",
    r"\bpastab\w*\b",
    r"\bgaut\w*\b",
    r"\bnegaut\w*\b",
    r"\bsvarst\w*\b",
    r"\bpristat\w*\b",
    r"\bdiskusij\w*\b",
    r"\bdiskutu\w*\b",
    r"\bpasisakym\w*\b",
    r"\bprotokol\w*\b",
    r"\bįraš\w*\b",
    r"\bskub\w*\b",
    r"\btaikom\w*\b",
    r"\bnetaikom\w*\b",
    r"\bnumatyt\w*\b",
    r"\bsuderin\w*\b",
    r"\bpakoreg\w*\b",
    r"\bįsigalioj\w*\b",
    r"\bstadij\w*\b",
    r"\btvark\w*\b",
    r"\breferendum\w*\b",
    r"\bpertrauk\w*\b",

    # Kreipiniai / mandagumo frazės
    r"\bkoleg\w*\b",
    r"\bgerbiam\w*\b",
    r"\bmiel\w*\b",
    r"\bačiū\b",
    r"\bdėk\w*\b",
    r"\blab(a|as)\s+(diena|rytas|vakaras)\b",
    r"\batsipraš\w*\b",

    # Laikas / datos
    r"\bval\b",
    r"\bm\b",
    r"\bd\b",
    r"\bmet\w*\b",
    r"\bdien\w*\b",
    r"\bryt\w*\b",
    r"\bvakar\w*\b",
    r"\bšiandien\w*\b",
    r"\b(pavasar|ruden|žiem|vasar)\w*\b",
    r"\b(saus|vasar|kov|balandž|geguž|biržel|liep|rugpjūč|rugsė|spal|lapkr|gruod)\w*\b",
    r"\b(pirmadien|antradien|trečiadien|ketvirtadien|penktadien|šeštadien|sekmadien)\w*\b",

    # Bendri stop žodžiai / įvardžiai
    r"\b(tiek|tam|tiem|tokiu|tokią|tokį|tokie|tokių|tas|tų|tuo|tokio|tokios|to|tos|tą|tie|todėl|tada|ten|tuos)\b",
    r"\b(šio|šių|šią|šį|šios|šioje|šiam|šito|šitą|šiuo|šiek|šitame|šiame)\b",
    r"\b(kitą|kitu|kitų|kiti|kuris|kurie|kurio|kuriuo|kuriuos|kuriam|kurių|kurios|kuria|kuri|kurį|kurią|ko|ką|kokio)\b",
    r"\b(jo|jos|juo|ja|jie|jį|ją|jų|jūsų|jam|jai|aš|mes|jūs|man|mums)\b",
    r"\b(viso|visi|visiems|visų|vieno|vieną)\b",
    r"\b(dėl|už|prieš|ir|nes|kad|kaip|kur|tai|į|iš|nuo|iki|per|po|apie|taip|bet|ar|o|bei)\b",

    # Pagalbiniai / menkai informatyvūs
    r"\b(būti|būtų|bus|buvo|nebuvo|nebūtų|nebus|yra|nebėra|nėra)\b",
    r"\bmanau\b",
    r"\bmanote\b",
    r"\btikrai\b",
    r"\btikr\w*\b",
    r"\blabai\b",
    r"\bjau\b",
    r"\bčia\b",
    r"\bdabar\b",
    r"\bgerai\b",
    r"\batrod\w*\b",
    r"\brod\w*\b",
    r"\baišku\b",
    r"\bmatyt\b",
    r"\bpavyzdžiui\b",
    r"\bkitaip\b",
    r"\bkartu\b",
    r"\bminėj\w*\b",
    r"\bapsisprend\w*\b",
    r"\bpasak\w*\b",
    r"\bgalvoj\w*\b",
    r"\batsakyk\w*\b",
    r"\bprikalb\w*\b",
    r"\bspekuliuoj\w*\b",

    # Balsavimo technika
    r"\bnari\w*\b",
    r"\bbiuleten\w*\b",
    r"\bgaliojanč\w*\b",
    r"\bnegaliojanč\w*\b",
    r"\brast\w*\b",
    r"\bišduot\w*\b",
    r"\bišrinkim\w*\b",

    # Instituciniai / procedūriniai likučiai
    r"\bvaldymo\b",
    r"\bvaldanči\w*\b",

    # Iš rezultatų matytas retorinis triukšmas
    r"\bekspromtu\b",
    r"\bneišgąsd\w*\b",
    r"\bnesuprat\w*\b",
    r"\bstovėj\w*\b",
    r"\bmatysim\w*\b",
    r"\bpasipuikuot\w*\b",
    r"\biškalb\w*\b",
    r"\bproaktyv\w*\b",
    r"\bkonstruktyv\w*\b",
    r"\bbendradarbiaut\w*\b",
    r"\bnetapt\w*\b",
    r"\bnebalsuot\w*\b",
    r"\bteko\b",
    r"\btekt\w*\b",
    r"\btrupuči\w*\b",
    r"\bnesumažin\w*\b",
    r"\bpadidint\w*\b",
    r"\bnieko\b",
    r"\bnenubyr\w*\b",
    r"\bpralaimėj\w*\b",
    r"\bpraeit\w*\b",
    r"\bnuogąstav\w*\b",
    r"\bnemažai\b",
    r"\bbūsim\w*\b",
    r"\bkritikuos\w*\b",
    r"\bpripaž\w*\b",
    r"\bnepatirt\w*\b",
    r"\bpasiged\w*\b",
    r"\bsužinoj\w*\b",
    r"\bnesielg\w*\b",
    r"\bužduo\w*\b",
    r"\bpaskatin\w*\b",
    r"\bnesprę\w*\b",
    r"\bnūdien\w*\b",
    r"\bmatys\w*\b",

    r"\bdžiaug\w*\b",
    r"\bpastebėj\w*\b",
    r"\bakcentuoj\w*\b",
    r"\bkreipi\w*\b",
    r"\bprisim\w*\b",
    r"\bsusimąst\w*\b",
    r"\bpasisak\w*\b",
    r"\bdiskutu\w*\b",
    r"\bargument\w*\b",

    r"\bsaulia\w*\b",
    r"\bkarbausk\w*\b",

    r"\bkažkok\w*\b",
    r"\bįsižeid\w*\b",
    r"\bneabejo\w*\b",
    r"\bpasiged\w*\b",
    r"\bneatrod\w*\b",
    r"\bnenuspėjam\w*\b",
    r"\babejing\w*\b",
    r"\bsugebė\w*\b",
    r"\bpasitais\w*\b",
    r"\bnuomon\w*\b",
    r"\btikiuos\w*\b",
    r"\bnežin\w*\b",
    r"\bpaaišk\w*\b",
    r"\bsuprant\w*\b",

    r"\batlik\w*\b",
    r"\bureguli\w*\b",
    r"\breglament\w*\b",
    r"\bužbaig\w*\b",
    r"\bsureag\w*\b",
    r"\bsurast\w*\b",
    r"\bsuteik\w*\b",
    r"\bpadary\w*\b",
    r"\bpradė\w*\b",
    r"\bkeit\w*\b",
    r"\bvykd\w*\b",
    r"\bįgyvend\w*\b",
    r"\batsispind\w*\b",
    r"\batsirem\w*\b",
    r"\bperplanu\w*\b",
    r"\bįvertin\w*\b",
    r"\bpatikslin\w*\b",
    r"\bpalaik\w*\b",
    r"\breagu\w*\b",

    r"\bskaiči\w*\b",
    r"\bkontrolin\w*\b",
    r"\bparametr\w*\b",
    r"\bprielaid\w*\b",
    r"\bformalum\w*\b",
    r"\bpatyrim\w*\b",
    r"\bstadij\w*\b",
    r"\bpirmin\w*\b",
    r"\btrimeč\w*\b",
    r"\basignavim\w*\b",

    r"\bvardu\b",
    r"\blydim\w*\b",
    r"\bpaket\w*\b",
    r"\bpateik\w*\b",
    r"\bsvarstym\w*\b",
    r"\bpriėmim\w*\b",
    r"\bpritart\w*\b",
    r"\bnepritart\w*\b",
    r"\bapsispręs\w*\b",
    r"\bapsisprend\w*\b",
    r"\bnukelt\w*\b",
    r"\bįtraukt\w*\b",
    r"\bperkelt\w*\b",
    r"\bpatvirtint\w*\b",
    r"\bskelb\w*\b",
    r"\bbendruomen\w*\b",
    r"\bišvad\w*\b",
    r"\blygmen\w*\b",
    r"\bprojekt\w*\b",
    r"\blydin\w*\b",
    r"\bpaket\w*\b",
    r"\bsiūlym\w*\b",
    r"\bpasiūl\w*\b",
    r"\bkomisij\w*\b",
    r"\bkomitet\w*\b",
    r"\bdepartament\w*\b",
    r"\bministerij\w*\b",
    r"\bįstaig\w*\b",
    r"\btarnyb\w*\b",
]

COMBINED_REGEX = re.compile("|".join(REGEX_PATTERNS), flags=re.IGNORECASE)

NR_REGEX = re.compile(r"\bNr\.\s*[A-Z0-9-]+\b", flags=re.IGNORECASE)
BRACKETS_REGEX = re.compile(r"\(.*?\)", flags=re.DOTALL)
ROMAN_REGEX = re.compile(r"\b[IVXLCDM]+\b", flags=re.IGNORECASE)
DIGITS_REGEX = re.compile(r"\d+")
BASIC_ALLOWED_REGEX = re.compile(r"[^a-ząčęėįšųūž\.\,\!\?\:\;\–\(\)\s]", flags=re.IGNORECASE)
SPACES_REGEX = re.compile(r"\s+")

SPEAKER_REGEX = re.compile(
    r"\b[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž\-]+\.?:?"
)

NAME_PAIR = r"[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž]+){1,2}"
NAME_LIST_REGEX = re.compile(rf"(?:{NAME_PAIR}\s+){{4,}}")

In [ ]:
def remove_speakers_and_name_lists(text):
    text = SPEAKER_REGEX.sub(" ", text)
    text = NAME_LIST_REGEX.sub(" ", text)
    return text


def regex_clean(text):
    text = NR_REGEX.sub(" ", text)
    text = COMBINED_REGEX.sub(" ", text)
    return SPACES_REGEX.sub(" ", text).strip()


def basic_clean(text):
    text = DIGITS_REGEX.sub(" ", text)
    text = BASIC_ALLOWED_REGEX.sub(" ", text)
    return text


def clean_punctuation_artifacts(text):
    text = re.sub(r"\s+[–-]\s*[.,!?;:]\s+", " ", text)
    text = re.sub(r"[.,!?;:]\s+[–-]\s*[.,!?;:]", " ", text)
    text = re.sub(r"\.\s+–\s+", ". ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"\s*–\s*", " – ", text)
    text = re.sub(r"([.,!?;:]){2,}", r"\1", text)
    return SPACES_REGEX.sub(" ", text).strip()


def stopwords_valymas(text, stopwords):
    return " ".join(
        w for w in text.split()
        if w not in stopwords
    )


def clean(text, stopwords):
    text = str(text)
    text = BRACKETS_REGEX.sub(" ", text)
    text = remove_speakers_and_name_lists(text)
    text = ROMAN_REGEX.sub(" ", text)
    text = text.lower()
    text = regex_clean(text)
    text = basic_clean(text)
    text = clean_punctuation_artifacts(text)
    text = stopwords_valymas(text, stopwords)
    text = SPACES_REGEX.sub(" ", text).strip()
    # Pašalina per trumpus tekstus
    if len(text.split()) <= 25:
        return np.nan

    return text

tqdm.pandas()

df1["clean_text"] = df1["skyrelis"].progress_apply(
    lambda x: clean(x, stopwords)
)

df2["clean_text"] = df2["skyrelis"].progress_apply(
    lambda x: clean(x, stopwords)
)

df1 = df1.dropna(subset=["clean_text"]).copy()
df2 = df2.dropna(subset=["clean_text"]).copy()

text_col = "clean_text"
extract_top_n = 20  
final_top_n = 3 

model_name = "EMBEDDIA/litlat-bert"

word_embedding_model = models.Transformer(
    model_name,
    max_seq_length=512
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

sentence_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device="cuda"
)

kw_model = KeyBERT(model=sentence_model)
tokenizer = sentence_model._first_module().tokenizer

In [ ]:
def split_into_sentences(text):
    text = str(text).strip()

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


def count_tokens(text):
    return (
        len(tokenizer.tokenize(str(text)))
        + tokenizer.num_special_tokens_to_add(pair=False)
    )


def split_into_chunks(text, max_tokens=480):
    sentences = split_into_sentences(text)

    chunks = []
    current_sentences = []

    for sent in sentences:
        sent_tokens = count_tokens(sent)

        if sent_tokens > max_tokens:
            if current_sentences:
                chunks.append(" ".join(current_sentences))
                current_sentences = []
            continue

        candidate = " ".join(current_sentences + [sent])
        candidate_tokens = count_tokens(candidate)

        if candidate_tokens <= max_tokens:
            current_sentences.append(sent)
        else:
            if current_sentences:
                chunks.append(" ".join(current_sentences))
            current_sentences = [sent]

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

In [ ]:
def clean_phrase_for_lemma(phrase):
    phrase = str(phrase).lower().strip()

    phrase = re.sub(
        r"[^\wąčęėįšųūž\s]",
        " ",
        phrase
    )

    phrase = re.sub(
        r"\s+",
        " ",
        phrase
    ).strip()

    return phrase


def analyze_phrase_spacy(phrase):
    phrase = clean_phrase_for_lemma(phrase)

    if not phrase:
        return "", (), False

    doc = nlp_spacy(phrase)

    lemmas = []
    pos_tags = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        lemma = token.lemma_.lower().strip()
        pos = token.pos_

        if lemma:
            lemmas.append(lemma)
        else:
            lemmas.append(token.text.lower())

        pos_tags.append(pos)

    lemma_key = " ".join(lemmas)
    pos_pattern = tuple(pos_tags)
    
    has_noun = any(
        p in {"NOUN", "PROPN"}
        for p in pos_tags
    )

    verb_count = sum(
        p in {"VERB", "AUX"}
        for p in pos_tags
    )

    length_ok = len(pos_tags) <= 4

    is_valid_pos = (
        has_noun
        and verb_count == 0
        and length_ok
    )

    return lemma_key, pos_pattern, is_valid_pos

In [ ]:
def extract_keybert_phrases(df, df_name, kadencija):
    results = []

    labels = df["pred_label"].dropna().unique()

    for label in tqdm(labels, desc=f"Processing {df_name}"):

        texts = (
            df.loc[df["pred_label"] == label, text_col]
            .dropna()
            .astype(str)
            .tolist()
        )

        if not texts:
            continue

        all_chunks = []

        for text in tqdm(
            texts,
            desc=f"Chunking | {df_name} | {label}",
            leave=False
        ):
            all_chunks.extend(
                split_into_chunks(
                    text,
                    max_tokens=480
                )
            )

        all_chunks = [
            chunk
            for chunk in all_chunks
            if chunk.strip()
        ]

        if not all_chunks:
            continue

        chunk_embeddings = sentence_model.encode(
            all_chunks,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=32
        )

        topic_center = chunk_embeddings.mean(axis=0)
        topic_center = topic_center / np.linalg.norm(topic_center)

        candidate_phrases = []

        for chunk in tqdm(
            all_chunks,
            desc=f"KeyBERT | {df_name} | {label}",
            leave=False
        ):
            keywords = kw_model.extract_keywords(
                chunk,
                keyphrase_ngram_range=(2, 3),
                stop_words=list(stopwords),
                top_n=extract_top_n,
                use_mmr=True,
                diversity=0.1
            )

            if not keywords:
                continue

            for phrase, keybert_score in keywords:
                candidate_phrases.append({
                    "fraze": phrase,
                    "keybert_score": keybert_score
                })

        if not candidate_phrases:
            continue

        temp_df = pd.DataFrame(candidate_phrases)

        phrase_df = (
            temp_df
            .groupby("fraze", as_index=False)
            .agg(
                keybert_score=("keybert_score", "mean"),
                pasikartojimai=("keybert_score", "count")
            )
        )

        phrases = phrase_df["fraze"].tolist()

        phrase_embeddings = sentence_model.encode(
            phrases,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=32
        )

        phrase_df["topic_center_score"] = phrase_embeddings @ topic_center

        phrase_df = (
            phrase_df
            .sort_values(
                by="topic_center_score",
                ascending=False
            )
            .head(250)
            .copy()
        )

        tqdm.pandas(desc=f"spaCy POS | {df_name} | {label}")

        phrase_analysis = (
            phrase_df["fraze"]
            .progress_apply(analyze_phrase_spacy)
        )

        phrase_df["lemma_key"] = phrase_analysis.apply(lambda x: x[0])
        phrase_df["pos_pattern"] = phrase_analysis.apply(lambda x: x[1])
        phrase_df["valid_pos"] = phrase_analysis.apply(lambda x: x[2])

        phrase_df = phrase_df[
            (phrase_df["lemma_key"].str.strip() != "")
            &
            (phrase_df["valid_pos"])
        ]

        phrase_df = (
            phrase_df
            .drop_duplicates(
                subset=["lemma_key"],
                keep="first"
            )
            .sort_values(
                by=["topic_center_score", "keybert_score", "pasikartojimai"],
                ascending=[False, False, False]
            )
            .head(final_top_n)
            .copy()
        )

        for rank, row in enumerate(phrase_df.itertuples(index=False), start=1):
            results.append({
                "dataset": df_name,
                "kadencija": kadencija,
                "tema": label,
                "rank": rank,
                "fraze": row.fraze,
                "lemma_key": row.lemma_key,
                "pos_pattern": row.pos_pattern,
                "keybert_score": round(row.keybert_score, 5),
                "topic_center_score": round(row.topic_center_score, 5),
                "pasikartojimai": row.pasikartojimai
            })

    return pd.DataFrame(results)

df1_keybert_phrases = extract_keybert_phrases(
    df1,
    "df1",
    "2016-2020"
)

df1_keybert_phrases.to_parquet(
    "2016-2020_keybert_phrases_top3.parquet",
    index=False
)

df2_keybert_phrases = extract_keybert_phrases(
    df2,
    "df2",
    "2020-2024"
)

df2_keybert_phrases.to_parquet(
    "2020-2024_keybert_phrases_top3.parquet",
    index=False
)

In [ ]:
# KONTEKSTAI

TOP_N_PHRASES = 2
TEXT_COL    = "tekstas"
CADENCE_COL = "kadencija"
MIN_WORDS   = 3

# Duomenys
df_sten = pd.read_parquet("seimo_stenogramos_2016_2024.parquet")

df_frazes = pd.concat([
    pd.read_parquet("/content/tfidf_frazes_2016_2020.parquet"),
    pd.read_parquet("/content/tfidf_frazes_2020_2024.parquet"),
], ignore_index=True)

print(f"Stenogramos:   {len(df_sten)} eilučių")
print(f"Frazės (viso): {len(df_frazes)} eilučių")
print(f"Kadencijos:    {sorted(df_frazes['kadencija'].unique())}")
print(f"Temos:         {sorted(df_frazes['tema'].unique())}")
print()

# Iš failo atrenkame po tris frazes temai
df_top = (
    df_frazes
    .sort_values("vieta")
    .groupby(["kadencija", "tema"], group_keys=False)
    .head(TOP_N_PHRASES)
    .reset_index(drop=True)
)

print(f"Pasirinktos frazės (TOP {TOP_N_PHRASES} kiekvienoje temoje):")
for _, r in df_top.iterrows():
    print(f"  [{r['kadencija']}] {r['tema']:40s}  vieta={r['vieta']}  '{r['fraze']}'")
print()

keyphrases = {}
for _, row in df_top.iterrows():
    keyphrases \
        .setdefault(row["kadencija"], {}) \
        .setdefault(row["tema"], []) \
        .append(row["fraze"])

In [ ]:
# Atliekamas minimalus teksto valymas
PARTY_PATTERNS = [
    r'Tėvynės\s+sąjungos?(?:\s*[\-–]\s*Lietuvos\s+krikščion\w+\s+demokrat\w+)?',
    r'Lietuvos\s+socialdemokrat\w+',
    r'Lietuvos\s+valstiečių\s+ir\s+žaliųjų\s+sąjungos?',
    r'Valstiečių\s+ir\s+žaliųjų\s+sąjungos?',
    r'Darbo\s+partij\w+',
    r'Liberalų\s+sąjūdžio?',
    r'Laisvės\s+partij\w+',
    r'Liberalų\s+ir\s+centro\s+sąjungos?',
    r'Lietuvos\s+lenkų\s+rinkimų\s+akcij\w+',
    r'Krikščionių\s+demokratų\s+sąjungos?',
    r'Lietuvos\s+žaliųjų\s+partij\w+',
    r'Lietuvos\s+centro\s+partij\w+',
    r'Demokratų\s+frakcij\w+',
]

def clean_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return text

    ABBREV_MAP = {
        r'\bNr\.':      'Nr<DOT>',
        r'\bstr\.':     'str<DOT>',
        r'\bskyr\.':    'skyr<DOT>',
        r'\bprof\.':    'prof<DOT>',
        r'\bdoc\.':     'doc<DOT>',
        r'\bdr\.':      'dr<DOT>',
        r'\bpan\.':     'pan<DOT>',
        r'\bpvz\.':     'pvz<DOT>',
        r'\bt\.y\.':    'ty<DOT>',
        r'\bt\.t\.':    'tt<DOT>',
        r'\bir\s+kt\.': 'ir kt<DOT>',
        r'\bkt\.':      'kt<DOT>',
        r'\bvnt\.':     'vnt<DOT>',
        r'\bproc\.':    'proc<DOT>',
        r'\bmln\.':     'mln<DOT>',
        r'\bmlrd\.':    'mlrd<DOT>',
        r'\btūkst\.':   'tūkst<DOT>',
        r'\bm\.\s*m\.': 'm<DOT> m<DOT>',
        r'(?<=\d)\s*m\.': ' m<DOT>',   # metai po skaičiaus: "2020 m."
        r'(?<=\d)\s*d\.': ' d<DOT>',   # diena po skaičiaus: "15 d."
        r'(?<=\d)\s*p\.': ' p<DOT>',   # puslapis
    }
    for pat, repl in ABBREV_MAP.items():
        text = re.sub(pat, repl, text)

    text = re.sub(r'\([^)]*\)', ' ', text)

    text = re.sub(
        r'\b[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽ][A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]{1,}\w*\b',
        ' ', text
    )

    text = re.sub(r'(?<!\w)[A-ZĄČĘĖĮŠŲŪŽ]\.(?!\w)', ' ', text)

    for pat in PARTY_PATTERNS:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)
        
    text = re.sub(r'\b[A-ZĄČĘĖĮŠŲŪŽ]{2,}\b', ' ', text)
    text = re.sub(r'\b(?:X{1,3}|IX|IV|VI{0,3}|II{0,2}|I{1,3})\b', ' ', text)
    text = re.sub(r'\b\d+(?:[.,]\d+)?\b', ' ', text)
    text = re.sub(r'(\w)\s+\.', r'\1.', text)
    text = re.sub(r'\.(\s*\.)+', '.', text)
    text = re.sub(r'^\s*\.\s*', '', text)
    text = re.sub(r'(?<!\w)[,;:\-–](?!\w)', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Skaidome į sakinius
def split_sentences(text: str) -> list:
    if pd.isna(text):
        return []
    text = str(text).replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r'(?<=[.!?])\s+(?=[A-ZĄČĘĖĮŠŲŪŽ„\"\u201E])', text)
    # <DOT> → '.' grąžiname čia, kiekviename sakinyje atskirai
    return [p.replace('<DOT>', '.').strip() for p in parts if p.strip()]

def word_count(sentence: str) -> int:
    return len(re.findall(r'\b\w{2,}\b', sentence))

def phrase_in_sentence(phrase: str, sentence: str) -> bool:
    return re.search(re.escape(phrase), sentence, flags=re.IGNORECASE) is not None


# Konteksto ištraukimo funkcija BERT modeliui
def get_bert_context(sentences: list, idx: int) -> str:
    """
    Grąžina kontekstą iš lygiai 3 sakinių:
      - tikslinis sakinys (sentences[idx])
      - po vieną sakinį prieš ir po jo (jei yra ir turi MIN_WORDS žodžių)

    Jei tiesioginis kaimynas per trumpas arba jo nėra,
    ieškoma toliau (iki 2 pozicijų) kad surinktume 3 pilnus sakinius.
    """
    n = len(sentences)

    def valid(s: str) -> bool:
        return word_count(s) >= MIN_WORDS

    prev = ""
    for offset in range(1, 3):
        if idx - offset < 0:
            break
        if valid(sentences[idx - offset]):
            prev = sentences[idx - offset]
            break

    nxt = ""
    for offset in range(1, 3):
        if idx + offset >= n:
            break
        if valid(sentences[idx + offset]):
            nxt = sentences[idx + offset]
            break

    parts = [s for s in [prev, sentences[idx], nxt] if s]
    return " ".join(parts).strip()

In [ ]:
# Konteksto ištraukimas
rows = []
total = len(df_sten)

for idx, row in df_sten.iterrows():
    cadence  = row[CADENCE_COL]
    raw_text = row[TEXT_COL]

    if cadence not in keyphrases:
        continue

    cleaned   = clean_text(raw_text)
    sentences = split_sentences(cleaned)

    for topic, phrases in keyphrases[cadence].items():
        for phrase in phrases:
            for i, sent in enumerate(sentences):
                if phrase_in_sentence(phrase, sent):

                    rows.append({
                        "cadence":          cadence,
                        "topic":            topic,
                        "phrase":           phrase,
                        "sentence_index":   i,
                        "vader_context_lt": sent,
                        "bert_context_lt":  get_bert_context(sentences, i),
                    })

    if (idx + 1) % 500 == 0:
        print(f"  Apdorota: {idx + 1}/{total}, kontekstai: {len(rows)}")

print(f"\nViso kontekstų: {len(rows)}")

contexts_df = pd.DataFrame(rows)
contexts_df.insert(0, "context_id", range(1, len(contexts_df) + 1))

# Patikriname kokybę
bert_same_as_vader = (contexts_df["bert_context_lt"] == contexts_df["vader_context_lt"]).sum()
sent_counts = contexts_df["bert_context_lt"].str.count(r'[.!?]')

print(f"\nKOKYBĖS ATASKAITA:")
print(f"  bert == vader (tik 1 sakinys):    {bert_same_as_vader} ({bert_same_as_vader/len(contexts_df)*100:.1f}%)")
print(f"  bert su 2+ skyrybos ženklais:     {(sent_counts >= 2).sum()} ({(sent_counts >= 2).sum()/len(contexts_df)*100:.1f}%)")
print(f"  Vidutinis bert ilgis (simboliai):  {contexts_df['bert_context_lt'].str.len().mean():.0f}")

In [ ]:
# Išsaugome kontekstus vertimui
out = Path("kontekstai_vertimui.parquet")
contexts_df.to_parquet(out, index=False)
print(f"\nIšsaugota: {out}  ({len(contexts_df)} eilučių)")
print(f"Stulpeliai: {list(contexts_df.columns)}")
print(contexts_df.head(3).to_string())

In [ ]:
# VERTIMAS

# Nustatome modelio pavadinimą ir failų kelius
MODEL_NAME = "Helsinki-NLP/opus-mt-tc-big-lt-en"

INPUT_FILE  = Path("/content/kontekstai_vertimui.parquet")
OUTPUT_FILE = Path("/content/kontekstai_isversti.parquet")

BATCH_SIZE = 32
MAX_LEN    = 512

#Tikrinamas GPU prieinamumas
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU nerastas")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Išvaloma atmintis prieš įkeliant modelį
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Duomenys
df = pd.read_parquet(INPUT_FILE)

print(f"\nĮkelta eilučių: {len(df)}")

all_lt_texts = pd.unique(
    pd.concat([
        df["vader_context_lt"],
        df["bert_context_lt"]
    ])
).tolist()

all_lt_texts = [
    t for t in all_lt_texts
    if isinstance(t, str) and t.strip()
]

print(f"Unikalių tekstų vertimui: {len(all_lt_texts)}")

In [ ]:
# Modelis
print(f"\nĮkeliamas modelis: {MODEL_NAME}")

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print("Modelis įkeltas")

# Vertimo funkcija
def translate_batch(texts):

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    del inputs, outputs

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return decoded

# Vertimas
def translate_all(texts):

    translation_map = {}
    total = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(total):

        batch = texts[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]

        print(f"Paketas {i+1}/{total} ({len(batch)} tekstų)", flush=True)

        try:
            translated = translate_batch(batch)
            for lt, en in zip(batch, translated):
                translation_map[lt] = en

        except Exception as e:
            print(f"KLAIDA: {e}")
            for lt in batch:
                translation_map[lt] = None

    return translation_map

print("\nVerčiama\n")

translation_map = translate_all(all_lt_texts)

df["vader_context_en"] = df["vader_context_lt"].map(translation_map)
df["bert_context_en"]  = df["bert_context_lt"].map(translation_map)

In [ ]:
# Rezultatai
ok      = sum(1 for v in translation_map.values() if v is not None)
missing = df["vader_context_en"].isna().sum()

print(f"\nIšversta: {ok}/{len(all_lt_texts)}")

if missing:
    print(f"Neišversta: {missing}")


# Pavyzdžiai
print("\nVERTIMŲ PAVYZDŽIAI:\n")

for _, row in df.dropna(subset=["vader_context_en"]).sample(3, random_state=42).iterrows():
    print("=" * 80)
    print("\n[LT]")
    print(row["vader_context_lt"])
    print("\n[EN]")
    print(row["vader_context_en"])


df.to_parquet(OUTPUT_FILE, index=False)
print(f"\nIšsaugota: {OUTPUT_FILE}")

In [ ]:
# SENTIMENTŲ ANALIZĖ

# Duomenys ir failų keliai
INPUT_FILE              = Path("kontekstai_isversti.parquet")
OUTPUT_INDIVIDUAL_FILE  = Path("sentimentai_kiekvienam_pasirodymui.parquet")
OUTPUT_SUMMARY_FILE     = Path("sentimentai_pagal_frazes.parquet")

df = pd.read_parquet(INPUT_FILE)

print(f"Įkelta: {len(df)} eilučių")
print(f"Stulpeliai: {list(df.columns)}")
print()

# VADER sentimentai
nltk.download("vader_lexicon", quiet=True)

vader = SentimentIntensityAnalyzer()

def vader_score(text) -> float:
    if pd.isna(text) or str(text).strip() == "":
        return np.nan
    return vader.polarity_scores(str(text))["compound"]

def sentiment_label(score) -> str:
    if pd.isna(score):
        return np.nan
    elif score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

print("Skaičiuojamas VADER sentimentas...")

df["vader_score"] = df["vader_context_en"].apply(vader_score)
df["vader_label"] = df["vader_score"].apply(sentiment_label)

print(f"  Paskirstymas:\n{df['vader_label'].value_counts().to_string()}")
print()

In [ ]:
# BERT sentimentai
device = 0 if torch.cuda.is_available() else -1

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Įkeliamas BERT modelis...")

bert_model = pipeline(
    "sentiment-analysis",
    model="phanerozoic/BERT-Sentiment-Classifier",
    device=device
)

texts = df["bert_context_en"].fillna("").astype(str).tolist()

batch_size = 128

bert_labels = []
bert_scores = []

print("Skaičiuojamas BERT sentimentas...")

for i in tqdm(range(0, len(texts), batch_size), desc="GPU sentiment"):

    batch = texts[i:i + batch_size]

    outputs = bert_model(
        batch,
        truncation=True,
        max_length=512
    )

    for o in outputs:

        label = o["label"]

        if label == "LABEL_1":
            label = "positive"
        elif label == "LABEL_0":
            label = "negative"

        bert_labels.append(label.lower())
        bert_scores.append(o["score"])

df["bert_label"] = bert_labels
df["bert_confidence"] = bert_scores

print(f"  Paskirstymas:\n{df['bert_label'].value_counts().to_string()}")
print()

In [ ]:
# Rezultatai kiekvienam pasirodymui
df.to_parquet(OUTPUT_INDIVIDUAL_FILE, index=False)

print(f"Išsaugota: {OUTPUT_INDIVIDUAL_FILE}  ({len(df)} eilučių)")


# Suvestinė pagal frazes
def bert_final_label(row) -> str:

    shares = {
        "positive": row["bert_positive_share"],
        "negative": row["bert_negative_share"],
    }

    return max(shares, key=shares.get)

phrase_summary = (
    df
    .groupby(["cadence", "topic", "phrase"])
    .agg(
        occurrences          = ("context_id", "count"),

        vader_mean           = ("vader_score", "mean"),
        vader_median         = ("vader_score", "median"),

        vader_positive_share = ("vader_label", lambda x: (x == "positive").mean()),
        vader_negative_share = ("vader_label", lambda x: (x == "negative").mean()),
        vader_neutral_share  = ("vader_label", lambda x: (x == "neutral").mean()),

        bert_positive_share  = ("bert_label", lambda x: (x == "positive").mean()),
        bert_negative_share  = ("bert_label", lambda x: (x == "negative").mean()),

        bert_avg_confidence  = ("bert_confidence", "mean"),
    )
    .reset_index()
)

phrase_summary["vader_final_label"] = (
    phrase_summary["vader_mean"]
    .apply(sentiment_label)
)

In [ ]:
# BERT sentimentas pagal daugumą
phrase_summary["bert_final_label"] = (
    phrase_summary.apply(bert_final_label, axis=1)
)

phrase_summary.to_parquet(OUTPUT_SUMMARY_FILE, index=False)

print(f"Išsaugota: {OUTPUT_SUMMARY_FILE}  ({len(phrase_summary)} frazių)")


# 6. Atvaizduojame rezultatus
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.3f}".format)

print()
print("=" * 80)
print("SENTIMENTŲ SUVESTINĖ PAGAL FRAZĘ")
print("=" * 80)

for cadence in sorted(phrase_summary["cadence"].unique()):

    print(f"\n── Kadencija: {cadence} ──")

    subset = phrase_summary[
        phrase_summary["cadence"] == cadence
    ].copy()

    display_cols = [
        "topic",
        "phrase",
        "occurrences",

        "vader_mean",
        "vader_final_label",

        "bert_positive_share",
        "bert_negative_share",

        "bert_final_label",
    ]

    print(subset[display_cols].to_string(index=False))

print()
print("=" * 80)
print("TRUMPA SANTRAUKA")
print("=" * 80)

print(f"\nIš viso analizuota frazių: {len(phrase_summary)}")
print(f"Iš viso kontekstų (pasirodymų): {phrase_summary['occurrences'].sum()}")

for cadence in sorted(phrase_summary["cadence"].unique()):

    sub = phrase_summary[
        phrase_summary["cadence"] == cadence
    ]

    print(f"\n{cadence}:")

    print(
        f"  VADER: "
        f"{(sub['vader_final_label']=='positive').sum()} teigiamos, "
        f"{(sub['vader_final_label']=='negative').sum()} neigiamos, "
        f"{(sub['vader_final_label']=='neutral').sum()} neutralios"
    )

    print(
        f"  BERT:  "
        f"{(sub['bert_final_label']=='positive').sum()} teigiamos, "
        f"{(sub['bert_final_label']=='negative').sum()} neigiamos"
    )
    
    

In [ ]:
# GRAFIKAI

# Nuskaitome failą
df = pd.read_parquet("sentimentai_pagal_frazes.parquet")

# Nustatome ka iš duomenų naudosime
TOP_N_TOPICS = 8
topics = sorted(df["topic"].unique())[:TOP_N_TOPICS]
cadences = sorted(df["cadence"].unique())

# Nustatome temų pavadinimus
TOPIC_LABELS = {
    "gynyba_ir_uzsienis": "Gynyba ir užsienio politika",
    "ekonomika_ir_mokesciai": "Ekonomika ir finansai",
    "socialine_politika": "Socialinė politika",
    "teise_ir_teisetvarka": "Teisė ir teisėtvarka",
    "vidaus_politika_ir_institucijos": "Vidaus politika ir institucijos",
    "aplinka_ir_zemes_ukis": "Aplinka ir žemės ūkis",
    "energetika_ir_infrastruktura": "Energetika ir infrastruktūra",
    "kultura_ir_istorija": "Kultūra ir istorija",
}

# Formatuojame y ašies procentus
def percent_no_minus(val, _):
    return f"{abs(val):.0f}%"


# Procentų korekcija, kad apvalinant neviršytų 100%
def fix_percentages(p_arr, n_arr, ng_arr):

    fixed_pos = []
    fixed_neu = []
    fixed_neg = []

    for p, n, ng in zip(p_arr, n_arr, ng_arr):

        vals = np.array([p, n, ng])

        # apvalinimas
        rounded = np.round(vals)

        # skirtumas iki 100
        diff = 100 - rounded.sum()

        # korekcija didžiausiam segmentui
        rounded[np.argmax(vals)] += diff

        fixed_pos.append(rounded[0])
        fixed_neu.append(rounded[1])
        fixed_neg.append(rounded[2])

    return (
        np.array(fixed_pos),
        np.array(fixed_neu),
        np.array(fixed_neg)
    )

In [ ]:
# Formatuojame grafikų panelę

def plot_panel(df, cadence, label_col, title):

    data = df[df["cadence"] == cadence]

    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.flatten()

    for i, topic in enumerate(topics):

        ax = axes[i]

        subset = data[data["topic"] == topic]

        if subset.empty:
            ax.set_title(
                TOPIC_LABELS.get(topic, topic),
                fontsize=10
            )
            ax.axis("off")
            continue

        
        # VADER grafikai
        if label_col == "vader_label":

            subset = subset.sort_values("vader_mean", ascending=False)

            phrases = subset["phrase"].values[:10]

            raw_pos = subset["vader_positive_share"].values[:10] * 100
            raw_neu = subset["vader_neutral_share"].values[:10] * 100
            raw_neg = subset["vader_negative_share"].values[:10] * 100

            pos, neu, neg_abs = fix_percentages(
                raw_pos,
                raw_neu,
                raw_neg
            )

            half_neu = neu / 2

            x = np.arange(len(phrases))
            min_display = 5

            ax.bar(x,  half_neu,                   color="khaki")
            ax.bar(x, -half_neu,                   color="khaki")

            ax.bar(
                x,
                pos,
                bottom=half_neu,
                color="lightgreen"
            )

            ax.bar(
                x,
                -neg_abs,
                bottom=-half_neu,
                color="lightcoral"
            )

            for xi, (p, n, ng) in enumerate(zip(pos, neu, neg_abs)):

                if p >= min_display:
                    ax.text(
                        xi,
                        half_neu[xi] + p / 2,
                        f"{p:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6,
                        fontweight="bold",
                        color="darkgreen"
                    )

                if n >= min_display:
                    ax.text(
                        xi,
                        0,
                        f"{n:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6,
                        fontweight="bold",
                        color="goldenrod"
                    )

                if ng >= min_display:
                    ax.text(
                        xi,
                        -half_neu[xi] - ng / 2,
                        f"{ng:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6,
                        fontweight="bold",
                        color="darkred"
                    )

        # BERT grafikai
        else:

            subset = subset.sort_values(
                "bert_positive_share",
                ascending=False
            )

            phrases = subset["phrase"].values[:10]

            pos = subset["bert_positive_share"].values[:10] * 100
            neg_abs = subset["bert_negative_share"].values[:10] * 100

            net = (
                subset["bert_positive_share"].values[:10]
                - subset["bert_negative_share"].values[:10]
            ) * 100

            x = np.arange(len(phrases))
            min_display = 5

            ax.bar(x, pos, color="lightgreen")

            ax.bar(x, -neg_abs, color="lightcoral")

            for xi, (p, ng) in enumerate(zip(pos, neg_abs)):

                if p >= min_display:
                    ax.text(
                        xi,
                        p / 2,
                        f"{p:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6,
                        fontweight="bold",
                        color="darkgreen"
                    )

                if ng >= min_display:
                    ax.text(
                        xi,
                        -ng / 2,
                        f"{ng:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6,
                        fontweight="bold",
                        color="darkred"
                    )

            ax.plot(
                x,
                net,
                color="black",
                linewidth=1.2,
                marker="o",
                markersize=4,
                zorder=5
            )

        # Formatuojame ašis ir etiketes
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(percent_no_minus)
        )

        ax.set_title(
            TOPIC_LABELS.get(topic, topic),
            fontsize=10
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            phrases,
            rotation=45,
            ha="right",
            fontsize=7
        )

        if label_col == "vader_label":
            ax.axhline(
                0,
                color="black",
                linewidth=0.8,
                zorder=0
            )
        else:
            ax.axhline(
                0,
                color="black",
                linewidth=0.8
            )

        ax.set_ylim(-100, 100)

    # Sudarome legendą
    if label_col == "vader_label":

        legend_elements = [
            Patch(facecolor="lightgreen", label="Teigiamas"),
            Patch(facecolor="khaki", label="Neutralus (centre)"),
            Patch(facecolor="lightcoral", label="Neigiamas"),
        ]

    else:

        legend_elements = [
            Patch(facecolor="lightgreen", label="Teigiamas"),
            Patch(facecolor="lightcoral", label="Neigiamas"),
            Line2D(
                [0], [0],
                color="black",
                marker="o",
                markersize=5,
                label="Sentimentų balansas"
            ),
        ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=4,
        fontsize=10,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.suptitle(title, fontsize=18)

    plt.tight_layout(rect=[0, 0.04, 1, 1])

    plt.show()

In [ ]:
# VADER panelės
for cadence in cadences:

    plot_panel(
        df=df,
        cadence=cadence,
        label_col="vader_label",
        title=f"VADER sentimentai | Kadencija {cadence}"
    )

# BERT panelės
for cadence in cadences:

    plot_panel(
        df=df,
        cadence=cadence,
        label_col="bert_label",
        title=f"BERT sentimentai | Kadencija {cadence}"
    )